# KOSIS ChromaDB 하이브리드 좌표 검색 → 2차 READY (Colab GPU)

1차 READY 이후의 ITEM/OBJ 좌표 후보를 ChromaDB dense + lexical + reranker 로 뽑고,
기존 KOSIS API 검증(`kosis_validate_mapping_candidates.py`)에 그대로 연결한다.

**원칙**: 임베딩/리랭커 점수는 후보 생성·순위에만 쓴다. READY 는 공식 메타 + KOSIS API 결과로만 확정한다.

**셀 순서**: 1 GPU 확인 → 2 저장소·의존성 → 3 Drive 마운트 → 4 입력 확인 →
5 Chroma 인덱스 생성 → 6 하이브리드 검색 → 7 API 검증 → 8 실제값 검증 →
9 A/B/C 평가 → 10 Drive 저장

In [ ]:
# 1. GPU 확인 (런타임 → 런타임 유형 변경 → GPU)
!nvidia-smi -L

In [ ]:
# 2. 저장소 + 의존성
!git clone https://github.com/rnwjdgus03/NLP_05-Team-Project-3.git repo
%cd repo
!git checkout codex/repro-baseline-20260727
!pip install -q -r requirements-ml.txt

In [ ]:
# 3. Drive 마운트 (기존 07_mapping_jinsung 결과 재사용)
from google.colab import drive, userdata
drive.mount('/content/drive')

import os
os.environ['KOSIS_API_KEY'] = userdata.get('KOSIS_API_KEY') or ''  # 코드/CSV/로그에 키를 남기지 않는다
if not os.environ['KOSIS_API_KEY']:
    raise RuntimeError('Colab 보안 비밀에 KOSIS_API_KEY를 등록하세요.')

RUN = '/content/drive/MyDrive/NLP_05-Team-Project-3/runs/contextual_top50_context_v2_8x3/07_mapping_jinsung'
OUT = RUN + '/chroma_hybrid'
!mkdir -p {OUT}
!ls {RUN}

In [ ]:
# 4. 입력 확인 (평가 대상 measurement 를 여기서 고정한다)
import pandas as pd
ready = pd.read_csv(f'{RUN}/05_hcx_measurements_kosis_ready.csv')
meta = pd.read_csv(f'{RUN}/05_hcx_measurements_kosis_meta_index.csv')
cand = pd.read_csv(f'{RUN}/05_hcx_measurements_kosis_table_candidates.csv')
print('1차 READY measurement:', ready['claim_measurement_id'].nunique())
print('meta rows:', len(meta), '| table candidates:', len(cand))

In [ ]:
# 5. Chroma 좌표 인덱스 생성 (BGE-M3 임베딩을 직접 저장 → manifest 로 모델·차원 고정)
!python kosis_build_chroma_meta_index.py \
  --meta-index {RUN}/05_hcx_measurements_kosis_meta_index.csv \
  --persist-dir data/indexes/kosis_meta_chroma \
  --collection kosis_meta_coordinates \
  --embedding-model BAAI/bge-m3 \
  --axis-value-limit 300 \
  --prd-se-source {RUN}/05_hcx_measurements_kosis_table_candidates.csv \
  --device cuda \
  --reset
!cat data/indexes/kosis_meta_chroma/chroma_manifest.json

In [ ]:
# 6. 하이브리드 검색 (metadata filter → dense → lexical → RRF → reranker → Top-10)
!python kosis_chroma_hybrid_search.py \
  --claims {RUN}/05_hcx_measurements_kosis_ready.csv \
  --table-candidates {RUN}/05_hcx_measurements_kosis_table_candidates.csv \
  --persist-dir data/indexes/kosis_meta_chroma \
  --collection kosis_meta_coordinates \
  --output {OUT}/05_hcx_measurements_kosis_chroma_candidates.csv \
  --stats-output {OUT}/chroma_search_stats.csv \
  --dense-top-k 50 --lexical-top-k 50 --rerank-top-k 20 --final-top-k 10 \
  --reranker-model BAAI/bge-reranker-v2-m3 --device cuda

### 6-1. (선택) 실험 B — Chroma dense 만
`--no-reranker --lexical-top-k 0` 으로 dense 단독 후보를 만들어 A/B/C 비교에 쓴다.

In [ ]:
!python kosis_chroma_hybrid_search.py \
  --claims {RUN}/05_hcx_measurements_kosis_ready.csv \
  --table-candidates {RUN}/05_hcx_measurements_kosis_table_candidates.csv \
  --persist-dir data/indexes/kosis_meta_chroma \
  --collection kosis_meta_coordinates \
  --output {OUT}/05_hcx_measurements_kosis_dense_only_candidates.csv \
  --dense-top-k 50 --lexical-top-k 0 --rerank-top-k 20 --final-top-k 10 \
  --no-reranker --device cuda

In [ ]:
# 7. 기존 KOSIS API 검증에 연결 (READY만 자동 확정, PROVISIONAL은 수동 검토)
!python kosis_validate_mapping_candidates.py \
  --input {OUT}/05_hcx_measurements_kosis_chroma_candidates.csv \
  --meta-index {RUN}/05_hcx_measurements_kosis_meta_index.csv \
  --output {OUT}/05_hcx_measurements_kosis_chroma_validated.csv \
  --evaluate-all-ranks \
  --strict-seeded-coordinate \
  --item-top-k 1 --obj-top-k 1 --max-combinations 1 \
  --allow-provisional

In [ ]:
# 7-1. measurement 단위 진단 (885 후보행 → measurement 단위로 축약)
!python diagnose_validated_mappings.py \
  --validated {OUT}/05_hcx_measurements_kosis_chroma_validated.csv \
  --output {OUT}/diagnosis_chroma.csv

In [ ]:
# 8. 실제값 검증 (기사일 컬럼이 있어야 REVISION_VINTAGE_RISK 정책이 동작)
import pandas as pd
validated = pd.read_csv(f'{OUT}/05_hcx_measurements_kosis_chroma_validated.csv')
ready = pd.read_csv(f'{RUN}/05_hcx_measurements_kosis_ready.csv')
if 'date' not in validated.columns and 'date' in ready.columns:
    validated = validated.merge(ready[['claim_measurement_id', 'date']],
                                on='claim_measurement_id', how='left')
validated[validated['mapping_status'] == 'READY'].to_csv(
    f'{OUT}/verify_input_chroma.csv', index=False, encoding='utf-8-sig')
print('verify 대상:', (validated['mapping_status'] == 'READY').sum())

In [ ]:
!python kosis_verify_claim_values.py \
  --input {OUT}/verify_input_chroma.csv \
  --output {OUT}/05_hcx_measurements_kosis_chroma_verified.csv \
  --delay 0.12

In [ ]:
# 9. A/B/C 동일 표본 평가 (골드 좌표가 없으면 recall 은 gold_required 로 표시된다)
GOLD = ''  # 예: f'{RUN}/gold_coordinates.csv'
gold_arg = f'--gold {GOLD}' if GOLD else ''

!python evaluate_chroma_hybrid_mapping.py --label A_baseline \
  --measurements {RUN}/05_hcx_measurements_kosis_ready.csv \
  --candidates {RUN}/05_hcx_measurements_kosis_candidates_with_meta.csv \
  --validated {RUN}/05_hcx_measurements_kosis_validated_mappings.csv \
  --verified {RUN}/05_hcx_measurements_kosis_verified.csv {gold_arg} \
  --output {OUT}/eval_A.json

!python evaluate_chroma_hybrid_mapping.py --label B_chroma_dense \
  --measurements {RUN}/05_hcx_measurements_kosis_ready.csv \
  --candidates {OUT}/05_hcx_measurements_kosis_dense_only_candidates.csv {gold_arg} \
  --output {OUT}/eval_B.json

!python evaluate_chroma_hybrid_mapping.py --label C_chroma_hybrid \
  --measurements {RUN}/05_hcx_measurements_kosis_ready.csv \
  --candidates {OUT}/05_hcx_measurements_kosis_chroma_candidates.csv \
  --validated {OUT}/05_hcx_measurements_kosis_chroma_validated.csv \
  --verified {OUT}/05_hcx_measurements_kosis_chroma_verified.csv \
  --stats {OUT}/chroma_search_stats.csv {gold_arg} \
  --output {OUT}/eval_C.json

In [ ]:
# 11. 원인 분리 — A는 API-valid 인데 C 가 놓친 measurement (골드 없이 검색 품질 측정)
#
# 아이디어: A 가 KOSIS API 로 코드 일치까지 확인한 좌표는 '사실상 정답에 가까운 좌표'다.
#          그 좌표가 C 의 Top-10 후보 안에 들어 있었는지 보면, 실패가
#          '검색이 못 찾은 것'인지 '순위에서 밀린 것'인지 '표부터 틀린 것'인지 갈린다.
import pandas as pd

TRUE = {"true", "1", "y", "yes", "t"}
DEEP = 3          # 엄격 비교 깊이 (obj_l1~l3)


def load(path):
    return pd.read_csv(path, dtype=str, keep_default_na=False)


def norm(v):
    v = str(v or "").strip()
    return "" if v.lower() in {"nan", "none"} else v


def api_ok(df, keys):
    if "response_code_valid" not in df.columns:
        raise KeyError("response_code_valid 없음 → 컬럼: " + ", ".join(list(df.columns)[:25]))
    flag = df["response_code_valid"].astype(str).str.strip().str.lower().isin(TRUE)
    return df[flag & df["claim_measurement_id"].isin(keys)]


def coord_cols(df):
    itm = "selected_itm_id" if "selected_itm_id" in df.columns else "itm_id"
    objs = []
    for i in range(1, 9):
        for col in (f"selected_obj_l{i}", f"obj_l{i}"):
            if col in df.columns:
                objs.append(col)
                break
    return itm, objs


def key_of(row, itm, objs, depth):
    return (norm(row.get("tbl_id")), norm(row.get(itm))) + tuple(
        norm(row.get(c)) for c in objs[:depth])


def rank_int(v):
    try:
        return int(float(v))
    except (TypeError, ValueError):
        return 999


A_val = load(f"{RUN}/05_hcx_measurements_kosis_validated_mappings.csv")
C_val = load(f"{OUT}/05_hcx_measurements_kosis_chroma_validated.csv")
C_cand = load(f"{OUT}/05_hcx_measurements_kosis_chroma_candidates.csv")
ready = load(f"{RUN}/05_hcx_measurements_kosis_ready.csv")
try:
    stats = load(f"{OUT}/chroma_search_stats.csv").drop_duplicates(
        "claim_measurement_id").set_index("claim_measurement_id")
except Exception as exc:
    print("stats 로드 실패:", exc)
    stats = None

KEYS = set(ready["claim_measurement_id"])
text_of = dict(zip(ready["claim_measurement_id"],
                   ready["claim_text"] if "claim_text" in ready.columns else [""] * len(ready)))

A_ok, C_ok = api_ok(A_val, KEYS), api_ok(C_val, KEYS)
A_set, C_set = set(A_ok["claim_measurement_id"]), set(C_ok["claim_measurement_id"])

print(f"평가 대상 {len(KEYS)} | A api-valid {len(A_set)} | C api-valid {len(C_set)}")
print(f"둘 다 {len(A_set & C_set)} | A만 {len(A_set - C_set)} | "
      f"C만 {len(C_set - A_set)} | 둘 다 실패 {len(KEYS - A_set - C_set)}")

itmA, objsA = coord_cols(A_val)
itmC, objsC = coord_cols(C_cand)
print("좌표 컬럼  A:", itmA, objsA[:DEEP], "| C:", itmC, objsC[:DEEP])

C_by_m = dict(tuple(C_cand.groupby("claim_measurement_id")))

rows = []
for mid in sorted(A_set):
    a = A_ok[A_ok["claim_measurement_id"] == mid]
    want_loose = {key_of(r, itmA, objsA, 1) for _, r in a.iterrows()}
    want_deep = {key_of(r, itmA, objsA, DEEP) for _, r in a.iterrows()}
    want_tbl = {k[0] for k in want_loose}
    g = C_by_m.get(mid)

    if g is None or g.empty:
        cls, hit_rank, deep_hit = "NO_CANDIDATE", "", False
    else:
        got_loose, got_deep, got_tbl = {}, set(), set()
        for _, r in g.iterrows():
            got_loose.setdefault(key_of(r, itmC, objsC, 1), rank_int(r.get("candidate_rank")))
            got_deep.add(key_of(r, itmC, objsC, DEEP))
            got_tbl.add(norm(r.get("tbl_id")))
        hits = [got_loose[k] for k in want_loose if k in got_loose]
        deep_hit = bool(want_deep & got_deep)
        if hits:
            cls, hit_rank = "COORD_IN_TOPK", min(hits)
        elif want_tbl & got_tbl:
            cls, hit_rank = "TABLE_OK_COORD_MISS", ""
        else:
            cls, hit_rank = "TABLE_MISS", ""

    st = stats.loc[mid].to_dict() if stats is not None and mid in stats.index else {}
    rows.append({
        "claim_measurement_id": mid,
        "c_api_valid": mid in C_set,
        "failure_class": cls,
        "coord_hit_rank": hit_rank,
        "deep_match": deep_hit,
        "a_tbl_id": " | ".join(sorted(want_tbl)),
        "a_coordinate": " | ".join(sorted("/".join(k[1:]) for k in want_deep))[:200],
        "c_candidate_rows": 0 if g is None else len(g),
        "dense_count": st.get("dense_count", ""),
        "lexical_count": st.get("lexical_count", ""),
        "claim_text": str(text_of.get(mid, ""))[:120],
    })

diag = pd.DataFrame(rows)
diag.to_csv(f"{OUT}/why_chroma_missed.csv", index=False, encoding="utf-8-sig")

print(f"\n=== A 의 API-valid 좌표가 C Top-10 안에 있었나 (분모 {len(diag)}) ===")
print(diag["failure_class"].value_counts().to_string())
loose = (diag["failure_class"] == "COORD_IN_TOPK").mean()
deep = diag["deep_match"].mean()
print(f"\n좌표 재현율 Top-10  느슨(tbl+itm+obj_l1) = {loose:.1%} | 엄격(obj_l3까지) = {deep:.1%}")
print("  ↑ 골드 없이 측정 가능한 검색 품질. 낮으면 '검색이 못 찾은 것'.")

top = diag[diag["failure_class"] == "COORD_IN_TOPK"]["coord_hit_rank"]
if len(top):
    print("\n맞춘 좌표의 rank 분포:")
    print(top.value_counts().sort_index().to_string())

miss = diag[~diag["c_api_valid"]]
print(f"\n=== A만 성공하고 C가 놓친 {len(miss)}건의 실패 유형 ===")
print(miss["failure_class"].value_counts().to_string())
print()
print(miss[["claim_measurement_id", "failure_class", "coord_hit_rank",
            "a_tbl_id", "claim_text"]].head(25).to_string(index=False))

no_cand = KEYS - set(C_cand["claim_measurement_id"])
print(f"\n=== C 가 후보를 아예 못 만든 measurement {len(no_cand)}건 ===")
for mid in sorted(no_cand):
    print(" ", mid, "|", str(text_of.get(mid, ""))[:90])
if no_cand and stats is not None:
    have = [m for m in sorted(no_cand) if m in stats.index]
    if have:
        print("\n해당 건 검색 통계 (필터가 후보를 다 걷어냈는지 확인):")
        print(stats.loc[have].to_string())

print(f"\n저장: {OUT}/why_chroma_missed.csv")


In [ ]:
# 12. 인덱스 커버리지 진단 — 정답 좌표가 '애초에 인덱스에 들어갔는가'
#
# 가설: 검색이 못 찾은 게 아니라, build_coordinates 의 두 상한 때문에
#       정답 좌표가 인덱스에 존재하지도 않았다.
#         (a) --axis-value-limit 40      축당 40개 값만
#         (b) --max-coordinates-per-table 4000
#             ↑ 이 상한은 item 루프 '바깥'에서 break 하므로,
#               앞쪽 item 이 4000개를 다 써버리면 뒤쪽 itm_id 는 좌표가 0개가 된다.
#
# build_coordinates 는 결정적이라 재실행하면 인덱스와 같은 결과가 나온다(재빌드 아님, GPU 불필요).
import json
from collections import defaultdict
from math import prod

import pandas as pd

from kosis_meta_coordinates import build_coordinates, group_meta_rows, read_csv_rows

TRUE = {"true", "1", "y", "yes", "t"}


def norm(v):
    v = str(v or "").strip()
    return "" if v.lower() in {"nan", "none"} else v


# ── 실행에 쓴 상한을 manifest 에서 그대로 읽는다 ───────────────────────────
man = {}
# 이번 실행으로 갓 만든 로컬 manifest 를 먼저 본다.
# Drive 사본은 '지난 실행'의 상한값일 수 있어 뒤로 미룬다.
for path in ("data/indexes/kosis_meta_chroma/chroma_manifest.json",
             f"{OUT}/kosis_meta_chroma_index/chroma_manifest.json"):
    try:
        man = json.load(open(path, encoding="utf-8"))
        print("manifest:", path)
        break
    except (FileNotFoundError, OSError):
        continue

AXIS_LIMIT = int(man.get("axis_value_limit", 40))
TABLE_CAP = int(man.get("max_coordinates_per_table", 4000))
print(f"axis_value_limit={AXIS_LIMIT} | max_coordinates_per_table={TABLE_CAP} | "
      f"manifest document_count={man.get('document_count')}")

# ── 메타에 '존재하는' 코드 vs 인덱스에 '들어간' 좌표 ──────────────────────
meta = read_csv_rows(f"{RUN}/05_hcx_measurements_kosis_meta_index.csv")
tables = group_meta_rows(meta)
print(f"메타 행 {len(meta):,} / 표 {len(tables)}개")

meta_itm, meta_obj1, n_items, axis_sizes = (defaultdict(set), defaultdict(set), {}, {})
for (_org, tbl), tab in tables.items():
    meta_itm[tbl] |= {i["code"] for i in tab["items"]}
    n_items[tbl] = len(tab["items"])
    axis_sizes[tbl] = {o: len(a["values"]) for o, a in sorted(tab["axes"].items())}
    if 1 in tab["axes"]:
        meta_obj1[tbl] |= {v["code"] for v in tab["axes"][1]["values"]}

coords = build_coordinates(meta, axis_value_limit=AXIS_LIMIT,
                           max_coordinates_per_table=TABLE_CAP)
print(f"재현된 좌표 {len(coords):,}개 (manifest 와 같아야 정상)")

cov_itm, cov_pair, made = defaultdict(set), defaultdict(set), defaultdict(int)
for c in coords:
    tbl = c["tbl_id"]
    made[tbl] += 1
    cov_itm[tbl].add(c["itm_id"])
    cov_pair[tbl].add((c["itm_id"], norm(c["obj_codes"].get(1, ""))))

# ── A 가 API 로 확인한 좌표를 단계별로 추적 ──────────────────────────────
diag = pd.read_csv(f"{OUT}/why_chroma_missed.csv", dtype=str, keep_default_na=False)
A_val = pd.read_csv(f"{RUN}/05_hcx_measurements_kosis_validated_mappings.csv",
                    dtype=str, keep_default_na=False)
A_ok = A_val[A_val["response_code_valid"].astype(str).str.strip().str.lower().isin(TRUE)]
A_ok = A_ok[A_ok["claim_measurement_id"].isin(set(diag["claim_measurement_id"]))]


def classify(tbl, itm, obj1):
    if tbl not in meta_itm:
        return "TBL_NOT_IN_META"          # 메타 인덱스 자체에 표가 없음
    if itm and itm not in meta_itm[tbl]:
        return "ITM_NOT_IN_META"          # 메타에 그 ITEM 코드가 없음
    if obj1 and meta_obj1.get(tbl) and obj1 not in meta_obj1[tbl]:
        return "OBJ_NOT_IN_META"
    if (itm, obj1) in cov_pair.get(tbl, set()):
        return "IN_INDEX"                 # 인덱스엔 있었다 → 필터/검색 문제
    if itm in cov_itm.get(tbl, set()):
        return "OBJ_TRUNCATED"            # ITEM 은 들어갔는데 그 축 조합이 잘림 (axis limit)
    return "ITM_TRUNCATED"                # ITEM 자체가 표 상한에 먹힘 (table cap)


rows = []
for _, r in A_ok.iterrows():
    tbl, itm = norm(r.get("tbl_id")), norm(r.get("selected_itm_id"))
    obj1 = norm(r.get("selected_obj_l1"))
    rows.append({
        "claim_measurement_id": r["claim_measurement_id"],
        "tbl_id": tbl, "itm_id": itm, "obj_l1": obj1,
        "coverage": classify(tbl, itm, obj1),
        "meta_items": n_items.get(tbl, 0),
        "indexed_items": len(cov_itm.get(tbl, set())),
        "coords_made": made.get(tbl, 0),
        "hit_table_cap": made.get(tbl, 0) >= TABLE_CAP,
        "axis_sizes": str(axis_sizes.get(tbl, {})),
    })

cov = pd.DataFrame(rows).drop_duplicates(["claim_measurement_id", "tbl_id", "itm_id", "obj_l1"])
cov = cov.merge(diag[["claim_measurement_id", "failure_class"]],
                on="claim_measurement_id", how="left")
cov.to_csv(f"{OUT}/index_coverage_diagnosis.csv", index=False, encoding="utf-8-sig")

print(f"\n=== 정답 좌표 {len(cov)}개가 인덱스에 들어갔는가 ===")
print(cov["coverage"].value_counts().to_string())
print("\n=== 검색 실패 유형 × 인덱스 커버리지 ===")
print(pd.crosstab(cov["failure_class"], cov["coverage"]).to_string())

# ── 표 상한이 ITEM 을 잡아먹는지 직접 확인 ────────────────────────────────
tbl_stat = (cov[["tbl_id", "meta_items", "indexed_items", "coords_made",
                 "hit_table_cap", "axis_sizes"]]
            .drop_duplicates("tbl_id").sort_values("meta_items", ascending=False))
tbl_stat["item_coverage"] = (tbl_stat["indexed_items"] /
                             tbl_stat["meta_items"].clip(lower=1)).round(3)
print(f"\n=== 실패에 관여한 표 {len(tbl_stat)}개의 ITEM 커버리지 ===")
print(tbl_stat.head(30).to_string(index=False))

capped = tbl_stat[tbl_stat["hit_table_cap"]]
print(f"\n표 상한({TABLE_CAP}) 도달: {len(capped)}/{len(tbl_stat)}개 표")
if len(tbl_stat):
    lost = tbl_stat["meta_items"].sum() - tbl_stat["indexed_items"].sum()
    print(f"이 표들의 ITEM {tbl_stat['meta_items'].sum():,}개 중 "
          f"{tbl_stat['indexed_items'].sum():,}개만 인덱싱 → {lost:,}개 누락")

# ── 상한을 올리면 얼마나 커지는지 (실제 빌드 전 비용 추정) ────────────────
print("\n=== 상한별 좌표 수 추정 (전체 표 기준) ===")
for limit in (40, 100, 300, 1000):
    total = 0
    for (_org, tbl), tab in tables.items():
        per_item = prod(min(len(a["values"]), limit) for a in tab["axes"].values()) or 1
        total += min(len(tab["items"]) * per_item, TABLE_CAP)
    print(f"  axis_value_limit={limit:>4} (표 상한 {TABLE_CAP} 유지) → 약 {total:,} 좌표")
for cap in (4000, 20000, 100000):
    total = 0
    for (_org, tbl), tab in tables.items():
        per_item = prod(min(len(a["values"]), AXIS_LIMIT) for a in tab["axes"].values()) or 1
        total += min(len(tab["items"]) * per_item, cap)
    print(f"  max_coordinates_per_table={cap:>6} (축 상한 {AXIS_LIMIT} 유지) → 약 {total:,} 좌표")

print(f"\n저장: {OUT}/index_coverage_diagnosis.csv")


In [ ]:
# 13. 어느 hard filter 술어가 정답 좌표를 잘랐는가 (추측 금지, 직접 재현)
#
# 셀 12 결론: 정답 좌표 85%가 인덱스에 있었는데 후보로 안 나왔다.
#             → build_chroma_where / passes_hard_filter 를 그대로 재현해 범인을 특정한다.
import json
from collections import Counter, defaultdict

import pandas as pd

from kosis_meta_coordinates import (build_coordinates, coordinate_metadata,
                                    prd_se_compatible, read_csv_rows,
                                    unit_dimension_compatible)
from kosis_chroma_hybrid_search import load_table_candidates

TRUE = {"true", "1", "y", "yes", "t"}
TOP_K_TABLES = 5          # 검색 때 쓴 값과 동일해야 한다


def norm(v):
    v = str(v or "").strip()
    return "" if v.lower() in {"nan", "none"} else v


def field(row, *names):
    for n in names:
        v = norm(row.get(n))
        if v:
            return v
    return ""


# ── 인덱스 좌표 재현 (셀 12 와 동일) ──────────────────────────────────────
meta = read_csv_rows(f"{RUN}/05_hcx_measurements_kosis_meta_index.csv")
prd_by_tbl = {}
for r in read_csv_rows(f"{RUN}/05_hcx_measurements_kosis_table_candidates.csv"):
    key = (field(r, "org_id", "ORG_ID"), field(r, "tbl_id", "TBL_ID"))
    val = field(r, "prd_se", "PRD_SE")
    if key[1] and val:
        prd_by_tbl.setdefault(key, val)

# 상한을 하드코딩하면 인덱스와 다른 좌표를 재현하게 된다 → manifest 에서 읽는다.
man = {}
for path in ("data/indexes/kosis_meta_chroma/chroma_manifest.json",
             f"{OUT}/kosis_meta_chroma_index/chroma_manifest.json"):
    try:
        man = json.load(open(path, encoding="utf-8"))
        break
    except (FileNotFoundError, OSError):
        continue
AXIS_LIMIT = int(man.get("axis_value_limit", 40))
TABLE_CAP = int(man.get("max_coordinates_per_table", 4000))
print(f"manifest 상한: axis_value_limit={AXIS_LIMIT} max_coordinates_per_table={TABLE_CAP}")

coords = build_coordinates(meta, axis_value_limit=AXIS_LIMIT,
                           max_coordinates_per_table=TABLE_CAP,
                           prd_se_by_table=prd_by_tbl)
print(f"좌표 재현 {len(coords):,}개 | 표별 prd_se 매핑 {len(prd_by_tbl)}개")

md_by_tbl = defaultdict(list)
md_by_key = {}
for c in coords:
    md = coordinate_metadata(c)
    md_by_tbl[md["tbl_id"]].append(md)
    md_by_key.setdefault((md["tbl_id"], md["itm_id"], md["obj_l1"]), md)

# ── 주장 / 상류 표 후보 ───────────────────────────────────────────────────
ready = {r["claim_measurement_id"]: r
         for r in read_csv_rows(f"{RUN}/05_hcx_measurements_kosis_ready.csv")}
tables = load_table_candidates(f"{RUN}/05_hcx_measurements_kosis_table_candidates.csv",
                               top_k=TOP_K_TABLES)

diag = pd.read_csv(f"{OUT}/why_chroma_missed.csv", dtype=str, keep_default_na=False)
fail_ids = set(diag.loc[diag["failure_class"] != "COORD_IN_TOPK", "claim_measurement_id"])
cls_of = dict(zip(diag["claim_measurement_id"], diag["failure_class"]))

A_val = pd.read_csv(f"{RUN}/05_hcx_measurements_kosis_validated_mappings.csv",
                    dtype=str, keep_default_na=False)
A_ok = A_val[A_val["response_code_valid"].astype(str).str.strip().str.lower().isin(TRUE)]
A_ok = A_ok[A_ok["claim_measurement_id"].isin(fail_ids)]

rows = []
for _, r in A_ok.iterrows():
    mid = r["claim_measurement_id"]
    claim = ready.get(mid, {})
    tbl, itm = norm(r.get("tbl_id")), norm(r.get("selected_itm_id"))
    obj1 = norm(r.get("selected_obj_l1"))

    top5 = [norm(t.get("tbl_id")) for t in tables.get(mid, [])]
    md = md_by_key.get((tbl, itm, obj1))

    claim_prd = field(claim, "measurement_prd_se", "prd_se")
    claim_dim = field(claim, "unit_dimension")
    mapping_type = field(claim, "mapping_type")
    coord_prd = md["prd_se"] if md else ""
    coord_dim = md["unit_dimension"] if md else ""

    in_top5 = tbl in top5
    prd_ok = prd_se_compatible(claim_prd, coord_prd) if md else None
    unit_ok = unit_dimension_compatible(claim_dim, coord_dim, mapping_type) if md else None

    # 주의: prd_se 는 2026-07-31 부터 hard filter 가 아니라 '순위 강등' 신호다.
    #       배제 사유로 세면 검색 실패를 필터 탓으로 잘못 돌리게 된다.
    if md is None:
        reason = "NOT_IN_INDEX"
    elif not in_top5:
        reason = "TBL_NOT_IN_TOP5"
    elif not unit_ok:
        reason = "UNIT_DIM_FILTER"
    elif not prd_ok:
        reason = "PASSED_FILTER_DEMOTED"  # 후보엔 남았고 뒤로 밀렸을 뿐
    else:
        reason = "PASSED_FILTER"          # 필터는 통과 → 검색/순위 문제

    # 해당 measurement 의 Top-5 표 전체에서 필터 생존 좌표 수
    # 생존 = 실제 hard filter 통과 (prd_se 는 배제하지 않으므로 제외)
    survivors = sum(
        1 for t in top5 for m in md_by_tbl.get(t, [])
        if unit_dimension_compatible(claim_dim, m["unit_dimension"], mapping_type))

    rows.append({
        "claim_measurement_id": mid,
        "failure_class": cls_of.get(mid, ""),
        "reject_reason": reason,
        "tbl_id": tbl, "in_top5": in_top5,
        "claim_prd_se": claim_prd, "coord_prd_se": coord_prd, "prd_ok": prd_ok,
        "claim_unit_dim": claim_dim, "coord_unit_dim": coord_dim,
        "mapping_type": mapping_type, "unit_ok": unit_ok,
        "top5_survivors": survivors,
        "top5_tbl_ids": " ".join(top5),
    })

f = pd.DataFrame(rows).drop_duplicates(["claim_measurement_id", "tbl_id"])
f.to_csv(f"{OUT}/which_filter_rejected.csv", index=False, encoding="utf-8-sig")

print(f"\n=== 실패 좌표 {len(f)}개의 원인 ===")
print("  (PASSED_FILTER* = 필터·인덱스 통과 → 순수 검색·순위 실패)")
print(f["reject_reason"].value_counts().to_string())
print("\n=== 실패 유형 × 자른 술어 ===")
print(pd.crosstab(f["failure_class"], f["reject_reason"]).to_string())

prd = f[f["reject_reason"] == "PASSED_FILTER_DEMOTED"]
if len(prd):
    print(f"\n=== prd_se 불일치로 '강등'된 {len(prd)}건 (배제 아님): 주장 주기 vs 좌표 주기 ===")
    print(Counter(zip(prd["claim_prd_se"], prd["coord_prd_se"])).most_common())

und = f[f["reject_reason"] == "UNIT_DIM_FILTER"]
if len(und):
    print(f"\n=== unit_dimension 으로 잘린 {len(und)}건 ===")
    print(Counter(zip(und["claim_unit_dim"], und["coord_unit_dim"])).most_common())

dead = f[f["top5_survivors"] == 0]
print(f"\n=== Top-5 표에서 생존 좌표가 0개인 measurement {dead['claim_measurement_id'].nunique()}건 ===")
print(dead[["claim_measurement_id", "claim_prd_se", "claim_unit_dim",
            "mapping_type", "top5_tbl_ids"]].head(20).to_string(index=False))

print("\n=== 필터를 껐다면 생존 좌표가 얼마나 늘었을까 (표본 평균) ===")
base = f["top5_survivors"].mean()
alt = []
for _, r in f.drop_duplicates("claim_measurement_id").iterrows():
    top5 = r["top5_tbl_ids"].split()
    alt.append(sum(len(md_by_tbl.get(t, [])) for t in top5))
print(f"  현재 필터 적용:  평균 {base:,.0f} 좌표")
print(f"  필터 전부 해제:  평균 {sum(alt)/max(len(alt),1):,.0f} 좌표")

print(f"\n저장: {OUT}/which_filter_rejected.csv")


In [ ]:
# 14. 실버 좌표 라벨 만들기 (골드 아님 — 검색 비교 전용)
#
# A/C 양쪽 후보 좌표를 KOSIS 실제값으로 대조해 '기사 숫자를 재현하는 좌표'를 찾는다.
# 좌표는 고정한 채(use_pinned_item) 조회하므로 A 후보 vs C 후보 비교가 성립한다.
#
# 한계(반드시 함께 읽을 것):
#   기사 숫자가 틀린 주장에서는 어떤 좌표도 재현하지 못해 라벨이 안 생긴다.
#   → 실버는 '참인 주장' 쪽으로 편향된다. 검색 비교에만 쓰고 판정 정확도엔 쓰지 말 것.
!python build_silver_coordinates.py \
  --measurements {RUN}/05_hcx_measurements_kosis_ready.csv \
  --candidates-a {RUN}/05_hcx_measurements_kosis_validated_mappings.csv \
  --candidates-c {OUT}/05_hcx_measurements_kosis_chroma_validated.csv \
  --output {OUT}/silver_coordinates.csv \
  --review-output {OUT}/needs_human_review.csv \
  --max-coordinates-per-measurement 12 \
  --delay 0.12


In [ ]:
# 15. 실버 좌표 위에서 A vs C 검색 재비교
#
# 분모가 SILVER_UNIQUE 로 확정된 measurement 로 줄어든다.
# 이 부분집합은 '기사 숫자가 KOSIS와 맞는' 쪽으로 편향돼 있으므로,
# 결과를 쓸 때 분모와 편향을 반드시 함께 적는다.
import pandas as pd

silver = pd.read_csv(f'{OUT}/silver_coordinates.csv', dtype=str, keep_default_na=False)
usable = silver[silver['tier'] == 'SILVER_UNIQUE']
print(f"실버 확정 {len(usable)}/{len(silver)} measurement")
print(silver['tier'].value_counts().to_string())

if len(usable) == 0:
    print("\n실버가 0건이라 비교할 수 없다. needs_human_review.csv 를 사람이 봐야 한다.")
else:
    gold_path = f'{OUT}/silver_as_gold.csv'
    usable.rename(columns={'silver_tbl_id': 'gold_tbl_id',
                           'silver_itm_id': 'gold_itm_id',
                           'silver_obj_l1': 'gold_obj_l1'})[
        ['claim_measurement_id', 'gold_tbl_id', 'gold_itm_id', 'gold_obj_l1']
    ].to_csv(gold_path, index=False, encoding='utf-8-sig')

    !python evaluate_chroma_hybrid_mapping.py --label A_baseline_silver \
      --measurements {RUN}/05_hcx_measurements_kosis_ready.csv \
      --candidates {RUN}/05_hcx_measurements_kosis_candidates_with_meta.csv \
      --validated {RUN}/05_hcx_measurements_kosis_validated_mappings.csv \
      --gold {gold_path} --output {OUT}/eval_A_silver.json

    !python evaluate_chroma_hybrid_mapping.py --label C_chroma_hybrid_silver \
      --measurements {RUN}/05_hcx_measurements_kosis_ready.csv \
      --candidates {OUT}/05_hcx_measurements_kosis_chroma_candidates.csv \
      --validated {OUT}/05_hcx_measurements_kosis_chroma_validated.csv \
      --stats {OUT}/chroma_search_stats.csv \
      --gold {gold_path} --output {OUT}/eval_C_silver.json

    import json
    for label in ('A', 'C'):
        d = json.load(open(f'{OUT}/eval_{label}_silver.json', encoding='utf-8'))
        print(f"\n[{d['label']}] 분모 {d['table_recall'].get('labeled')}건")
        for name in ('table_recall', 'item_recall', 'obj_recall'):
            print(f"  {name}: {d[name]}")

    print("\n[경고] 위 recall 의 분모는 '기사 숫자가 KOSIS와 맞은' measurement 뿐이다.")
    print("       A팀 골드가 오면 실버와 얼마나 일치했는지부터 대조할 것.")


In [ ]:
# 16. 라벨링 근거 시트 뽑기 (실버가 자동 확정 못 한 measurement)
#
# 라벨러가 prior 가 아니라 '증거'를 보고 고르게 한다:
#   후보 좌표 + KOSIS 메타 이름/단위 + 실제 조회값 + 판정
# 후보는 파이프라인 순위가 아니라 코드 사전순으로 섞어 낸다(앵커링 방지).
!python export_labeling_packet.py \
  --silver {OUT}/silver_coordinates.csv \
  --review {OUT}/needs_human_review.csv \
  --measurements {RUN}/05_hcx_measurements_kosis_ready.csv \
  --output {OUT}/labeling_packet.csv \
  --markdown {OUT}/labeling_packet.md

# 채팅에 붙여넣을 배치 (한 번에 30건씩)
import pandas as pd
pk = pd.read_csv(f'{OUT}/labeling_packet.csv', dtype=str, keep_default_na=False)
print(f"\n라벨 필요 {len(pk)}건 → 30건씩 {-(-len(pk)//30)}배치")
print(pk['tier'].value_counts().to_string())
print("\n--- 배치 1 미리보기 ---")
md = open(f'{OUT}/labeling_packet.md', encoding='utf-8').read()
print(md[:1500])


In [ ]:
# 17. KOSIS 에러코드 분포 (분기 전략을 짜기 전에 '실제로 뭐가 나오는지' 먼저 센다)
#
# 현재 코드는 err:30 만 빈응답으로 처리하고 나머지는 전부 조합을 포기한다.
#   err:20 세부항목 누락  → objL 차원을 단계적으로 열면 회수 가능
#   err:31 응답 too large → 요청을 좁혀야 함
# 이 둘이 실제로 얼마나 나오는지 확인하고 나서 구현한다.
import re
from collections import Counter

import pandas as pd

frames = {
    "A_baseline": f"{RUN}/05_hcx_measurements_kosis_validated_mappings.csv",
    "C_chroma": f"{OUT}/05_hcx_measurements_kosis_chroma_validated.csv",
}

for label, path in frames.items():
    try:
        df = pd.read_csv(path, dtype=str, keep_default_na=False)
    except FileNotFoundError:
        print(f"[{label}] 파일 없음: {path}")
        continue

    col = "api_error" if "api_error" in df.columns else None
    print(f"\n=== {label} | 행 {len(df):,} ===")
    if col is None:
        print("  api_error 컬럼 없음 → 컬럼:", ", ".join(list(df.columns)[:25]))
        continue

    errors = df[df[col].str.strip() != ""][col]
    print(f"  KOSIS 에러가 기록된 행: {len(errors):,} ({len(errors)/max(len(df),1):.1%})")

    codes = Counter()
    for value in errors:
        found = re.search(r"KOSIS_ERROR\[([^\]]*)\]", value)
        if found:
            for code in found.group(1).split(","):
                codes[code.strip()] += 1
        else:
            codes["(형식 불명)"] += 1
    if codes:
        print("  에러코드별:")
        for code, n in codes.most_common():
            print(f"    err:{code} → {n:,}")

    # 메시지 표본 (코드별 1건씩)
    seen = set()
    for value in errors:
        found = re.search(r"KOSIS_ERROR\[([^\]]*)\]", value)
        key = found.group(1) if found else "?"
        if key not in seen:
            seen.add(key)
            print(f"    예시[{key}]: {value[:160]}")

    # 에러 말고 '빈 응답'은 몇 건인가 (분기 대상 규모 비교용)
    if "validation_reason" in df.columns:
        reasons = df["validation_reason"].str.strip()
        print("  validation_reason 상위:")
        for reason, n in Counter(r for r in reasons if r).most_common(8):
            print(f"    {reason}: {n:,}")

print("\n판단 기준:")
print("  err:20 이 유의미하게 나오면 → objL 단계적 확장 구현 (ITEM_OBJ_FIXABLE 회수 기대)")
print("  err:20/31 이 거의 없으면   → 분기 전략은 우리 데이터에선 효과 없음. 구현하지 말 것")


In [ ]:
# 18. 상류(추출) 품질이 검색 실패의 원인인가
#
# 지금까지 진단은 '주장 쪽은 옳다'고 가정하고 검색만 채점했다.
# 주장 텍스트가 잘렸거나, 한 문장을 measurement 여러 개로 쪼갰거나,
# KOSIS 에 없는 대상(개별 브랜드 상품가 등)이면 어떤 검색기도 성공할 수 없다.
!python diagnose_claim_quality.py \
  --measurements {RUN}/05_hcx_measurements_kosis_ready.csv \
  --retrieval {OUT}/why_chroma_missed.csv \
  --output {OUT}/claim_quality_diagnosis.csv


In [ ]:
# 19. 새 범위 게이트를 기존 177건에 적용해보기 (파이프라인 재실행 없이 dry-run)
#
# 검증 방법: 게이트가 막은 건이 실버 NO_MATCH 에 몰려 있어야 한다.
#   - NO_MATCH 를 잘 막으면  → 게이트가 제 일을 한 것
#   - SILVER_UNIQUE 를 막으면 → 오탐. 규칙을 좁혀야 한다 (이쪽이 더 나쁘다)
from collections import Counter

import pandas as pd

from kosis_scope_gate import gate_decision

ready = pd.read_csv(f'{RUN}/05_hcx_measurements_kosis_ready.csv',
                    dtype=str, keep_default_na=False)
decisions = pd.DataFrame([gate_decision(r) for r in ready.to_dict('records')])
ready = pd.concat([ready.reset_index(drop=True), decisions], axis=1)
ready.to_csv(f'{OUT}/scope_gate_dryrun.csv', index=False, encoding='utf-8-sig')

blocked = ready[ready['scope_gate_blocked'] == 'Y']
review = ready[ready['scope_gate_severity'] == 'REVIEW']
print(f"1차 READY {len(ready)}건")
print(f"  새 게이트가 차단: {len(blocked)} ({len(blocked)/max(len(ready),1):.1%})")
print(f"  사람 확인 필요:   {len(review)}")
print(f"  남는 건:          {len(ready) - len(blocked)}")
print("\n차단 사유별:")
print(Counter(blocked['scope_gate_code']).most_common())

# --- 검증: 실버 tier 와 교차 ---
try:
    silver = pd.read_csv(f'{OUT}/silver_coordinates.csv', dtype=str, keep_default_na=False)
    merged = ready.merge(silver[['claim_measurement_id', 'tier']],
                         on='claim_measurement_id', how='left')
    print("\n=== 실버 tier × 새 게이트 (핵심 검증) ===")
    print(pd.crosstab(merged['tier'].fillna('(없음)'),
                      merged['scope_gate_blocked']).to_string())

    wrong = merged[(merged['scope_gate_blocked'] == 'Y') &
                   (merged['tier'] == 'SILVER_UNIQUE')]
    print(f"\n오탐(값이 재현됐는데 차단): {len(wrong)}건  ← 0이어야 한다")
    for _, r in wrong.iterrows():
        print(f"  {r['claim_measurement_id']} [{r['scope_gate_code']}]")
        print(f"    {str(r['claim_text'])[:110]}")

    nomatch = merged[merged['tier'] == 'NO_MATCH']
    caught = (nomatch['scope_gate_blocked'] == 'Y').sum()
    print(f"\nNO_MATCH {len(nomatch)}건 중 게이트가 잡은 것: {caught} "
          f"({caught/max(len(nomatch),1):.1%})")
except FileNotFoundError:
    print("\n(silver_coordinates.csv 없음 — 교차 검증 건너뜀)")

print("\n=== 차단된 주장 표본 ===")
for _, r in blocked.head(20).iterrows():
    print(f"  [{r['scope_gate_code']}] {str(r['claim_text'])[:100]}")
    print(f"      근거: {r['scope_gate_reason'][:90]}")

print(f"\n저장: {OUT}/scope_gate_dryrun.csv")


In [ ]:
# 10. 결과 저장
# 좌표 12만 기준 Chroma 인덱스는 0.5~1GB 라 Drive 복사가 오래 걸린다.
# build_coordinates 는 결정적이므로 인덱스는 언제든 재생성 가능 → 기본은 복사하지 않는다.
# (Git 에는 어떤 경우에도 커밋하지 않는다.)
SAVE_INDEX_TO_DRIVE = False

!du -sh data/indexes/kosis_meta_chroma
if SAVE_INDEX_TO_DRIVE:
    !cp -r data/indexes/kosis_meta_chroma {OUT}/kosis_meta_chroma_index
else:
    # manifest 만 남긴다 (재현에 필요한 모델·차원·해시·상한값이 여기 다 있다)
    !mkdir -p {OUT}/kosis_meta_chroma_index
    !cp data/indexes/kosis_meta_chroma/chroma_manifest.json {OUT}/kosis_meta_chroma_index/
    print('인덱스 본체는 저장하지 않음 (SAVE_INDEX_TO_DRIVE=True 로 바꾸면 복사)')

!ls -la {OUT}


## 해석 주의
- 후보행 수(measurement × Top-K)를 measurement 실패 건수로 읽지 말 것.
- 골드 좌표(`gold_tbl_id` / `gold_itm_id` / `gold_obj_l1`)가 없으면 recall 은 계산하지 않는다.
- READY 증가만으로 개선을 주장하지 말고, 동일 표본 A/B/C 결과와 수동 검수 Precision 으로 판단한다.